# 02 — Prior Predictive Check

Sample parameters from the prior and run SIPNET in parallel via PyEns.
Plot a fan chart of NEE timeseries and verify the prior is appropriately wide.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path

from pysipnet import SIPNETModel, SIPNETRunner, ClimateDrivers
from pysipnet.runner import ClimateStaging
from pysipnet.parameters import (
    SIPNETParametersV1, InitialConditions, PhotosynthesisParams,
    PhenologyParams, RespirationParams, AllocationParams, WaterParams,
    LeafPhysiologyParams,
)
from sipnet_calibration import build_prior, prior_predictive


In [ ]:
BASE_PARAMS = SIPNETParametersV1(
    initial_conditions=InitialConditions(
        plant_wood=30000.0, lai=0.0, soil=10000.0, soil_water_frac=0.5,
        snow=1.0, fine_root_frac=0.05, coarse_root_frac=0.15,
    ),
    photosynthesis=PhotosynthesisParams(
        a_max=120.0, a_max_frac=0.76, base_fol_resp_frac=0.1,
        psn_t_min=2.0, psn_t_opt=20.0, d_vpd_slope=0.05, d_vpd_exp=1.0,
        half_sat_par=200.0, attenuation=0.5,
    ),
    phenology=PhenologyParams(
        leaf_off_day=270.0, gdd_leaf_on=100.0, leaf_growth=50.0,
        frac_leaf_fall=0.95, leaf_allocation=0.25, leaf_turnover_rate=1.0,
    ),
    respiration=RespirationParams(
        base_veg_resp=0.5, veg_resp_q10=2.0, growth_resp_frac=0.0,
        frozen_soil_fol_r_eff=0.5, frozen_soil_threshold=-1.0,
        base_fine_root_resp=0.5, base_coarse_root_resp=0.1,
        fine_root_q10=2.0, coarse_root_q10=2.0,
        base_soil_resp=0.3, soil_resp_q10=2.2, soil_resp_moist_effect=1.5,
    ),
    allocation=AllocationParams(
        fine_root_allocation=0.35, wood_allocation=0.30,
        fine_root_turnover_rate=1.0, coarse_root_turnover_rate=0.1,
        wood_turnover_rate=0.02,
    ),
    water=WaterParams(
        water_remove_frac=0.1, frozen_soil_eff=0.1, wue_const=10.0, soil_whc=12.0,
        litter_whc=5.0, immed_evap_frac=0.1, fast_flow_frac=0.1, snow_melt=0.15,
        rd_const=100.0, r_soil_const1=3.0, r_soil_const2=2.0,
    ),
    leaf=LeafPhysiologyParams(leaf_c_sp_wt=32.0, c_frac_leaf=0.45),
)

GROUND_TRUTH = {
    "a_max":          120.0,
    "psn_t_opt":       20.0,
    "half_sat_par":   200.0,
    "base_veg_resp":    0.5,
    "veg_resp_q10":     2.0,
    "base_soil_resp":   0.3,
    "soil_resp_q10":    2.2,
    "wue_const":       10.0,
}
SIGMA_OBS = 2.0  # gC m-2 day-1

PARAM_SPECS = {
    "a_max":          {"prior_type": "lognormal", "loc": float(jnp.log(120.0)), "scale": 0.5},
    "psn_t_opt":      {"prior_type": "normal",    "loc": 20.0,                  "scale": 8.0},
    "half_sat_par":   {"prior_type": "lognormal", "loc": float(jnp.log(200.0)), "scale": 0.5},
    "base_veg_resp":  {"prior_type": "lognormal", "loc": float(jnp.log(0.5)),   "scale": 0.5},
    "veg_resp_q10":   {"prior_type": "lognormal", "loc": float(jnp.log(2.0)),   "scale": 0.3},
    "base_soil_resp": {"prior_type": "lognormal", "loc": float(jnp.log(0.3)),   "scale": 0.5},
    "soil_resp_q10":  {"prior_type": "lognormal", "loc": float(jnp.log(2.2)),   "scale": 0.3},
    "wue_const":      {"prior_type": "lognormal", "loc": float(jnp.log(10.0)),  "scale": 0.5},
}
PARAM_NAMES = list(PARAM_SPECS.keys())


In [ ]:
climate = ClimateDrivers.from_path(Path("../data/benchmark_climate.clim"))
runner = SIPNETRunner(climate_staging=ClimateStaging.SYMLINK)
model = SIPNETModel(runner, base_params=BASE_PARAMS, base_climate=climate)

prior = build_prior(PARAM_SPECS)
print("Prior fields (alphabetical):", prior.record_template.fields)
print("Flat size:", prior.record_template.flat_size)


In [ ]:
prior_nee = prior_predictive(prior, model, PARAM_NAMES, n_samples=200, n_workers=4, seed=0)
print(f"Prior predictive shape: {prior_nee.shape}")


In [ ]:
# Ground truth NEE for reference
truth_nee = model(**GROUND_TRUTH).nee().values
T = prior_nee.shape[1]
t = np.arange(T)

pcts = np.percentile(prior_nee, [5, 25, 50, 75, 95], axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(t, pcts[0], pcts[4], alpha=0.2, color="C0", label="5–95%")
ax.fill_between(t, pcts[1], pcts[3], alpha=0.4, color="C0", label="25–75%")
ax.plot(t, pcts[2], color="C0", lw=1.5, label="Prior median")
ax.plot(t, truth_nee, color="red", lw=1.5, label="Ground truth", zorder=5)
ax.set_xlabel("Timestep")
ax.set_ylabel("NEE (gC m$^{-2}$ day$^{-1}$)")
ax.set_title("Prior predictive fan chart")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Ground truth NEE range: [{truth_nee.min():.2f}, {truth_nee.max():.2f}]")
print(f"Prior 5–95% range: [{pcts[0].min():.2f}, {pcts[4].max():.2f}]")


In [ ]:
# Marginal prior distributions for each parameter
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

import jax
key = jax.random.PRNGKey(99)
prior_samples = prior._sample(key, (2000,))

for i, name in enumerate(PARAM_NAMES):
    ax = axes[i]
    vals = np.array(prior_samples[name])
    ax.hist(vals, bins=40, density=True, alpha=0.7, color="C0")
    ax.axvline(GROUND_TRUTH[name], color="red", lw=2, label="truth")
    ax.set_title(name)
    ax.set_xlabel("Value")
axes[0].legend()
plt.suptitle("Prior marginal distributions (red = ground truth)")
plt.tight_layout()
plt.show()
